## The runtime layer — `containerd`, `runc` & alternatives

Back to the very center of the map. From module 01: the daemon **doesn't run containers itself**. `dockerd` hands off to **`containerd`** (the runtime supervisor, which tracks lifecycle) which drives **`runc`** (the low-level OCI runtime that does the namespace/cgroup/exec dance and becomes the container's PID 1). You almost never touch this layer — but knowing it's swappable explains a class of advanced setups.

`runc` is one of several **OCI runtimes**, interchangeable because they share a standard interface:

- **`crun`** — a C reimplementation of `runc` that starts containers ~10× faster and uses less memory. A low-risk drop-in; set `default-runtime: crun`.
- **`kata-runtime`** — runs each container inside a **lightweight VM** for *real hardware isolation*. Heavier, but it fixes the "container is not a security boundary" problem from module 08 for multi-tenant clusters.
- **`gvisor` / `runsc`** — a **userspace kernel**: it reimplements much of the Linux syscall surface in Go, intercepting the container's syscalls before they reach the host kernel. Strong isolation at a modest performance cost.

Register one in `daemon.json` and select it per container:

```bash
docker run --runtime=kata-runtime alpine sh
```

The bigger picture: **`containerd` is a standalone project**, not Docker-only. **Kubernetes talks to it directly** through the CRI (Container Runtime Interface), skipping `dockerd` entirely — which is what the "Kubernetes deprecated Docker" headlines really meant (they dropped the `dockerd` shim, not your images). Your OCI images run identically on `containerd` under Docker or under Kubernetes. Understanding this stack — daemon → containerd → runtime → kernel — is what lets you reason about performance, isolation strength, and how Docker relates to the wider container ecosystem.